In [5]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client=Client(cluster)

2025-07-22 15:48:17,179 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:61873 (pid=9682) exceeded 95% memory budget. Restarting...
2025-07-22 15:48:17,259 - distributed.nanny - WARNING - Restarting worker
2025-07-22 15:48:55,282 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:61871 (pid=9683) exceeded 95% memory budget. Restarting...
2025-07-22 15:48:55,400 - distributed.nanny - WARNING - Restarting worker


In [8]:
#load data
path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"
primordial=scm.ortho.load(client,path,name)

In [9]:
#create a simulation batch object
batch=scm.simulation_batch(primordial)
batch.describe_primordial(client)

In [11]:
z=scm.simulate_from_description(batch.description_primordial_by_cre)

In [13]:
z.compute()

2025-07-22 15:48:53,317 - distributed.worker - WARNING - Compute Failed
Key:       ('simulate_partition-d6e0aa96d332fe361fda6ad61148beec', 15)
Function:  subgraph_callable-4ddc6e1a-c016-4a23-86c1-97383347
args:      (                cre_id rep_id  ... sigmasquare         p
5478  Lamc1_chr1_12183    2B1  ...         NaN       NaN
5479  Lamc1_chr1_12183    2B2  ...         NaN       NaN
5480  Lamc1_chr1_12183     A1  ...         NaN       NaN
5481  Lamc1_chr1_12183     A2  ...         NaN       NaN
5482  Lamc1_chr1_12183     B1  ...         NaN       NaN
...                ...    ...  ...         ...       ...
5839  Map1b_chr13_9447     B1  ...    0.997996  0.359264
5840  Map1b_chr13_9447     B1  ...    1.108135  0.344776
5841  Map1b_chr13_9447     B1  ...    0.654022  0.421640
5842  Map1b_chr13_9447     B1  ...    0.951995  0.365925
5843  Map1b_chr13_9447     B1  ...    0.643780  0.424082

[366 rows x 10 columns])
kwargs:    {}
Exception: "ValueError('p < 0, p > 1 or p contains NaNs')"


ValueError: p < 0, p > 1 or p contains NaNs

2025-07-22 15:48:55,037 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 2.63 GiB -- Worker memory limit: 3.43 GiB


In [ ]:
df=batch.description_primordial_by_cre.compute()
any(df["r"].isnull())
any(df["p"].isnull())


In [ ]:
repeated_df = df.loc[df.index.repeat(df['cells'])].reset_index(drop=True)

In [10]:
batch.simulate_many(client,3)

2025-07-22 15:48:16,024 - distributed.worker - WARNING - Compute Failed
Key:       ('simulate_partition-d6e0aa96d332fe361fda6ad61148beec', 15)
Function:  subgraph_callable-e944892e-2880-494a-96e2-f703bcfe
args:      (                cre_id rep_id  ... sigmasquare         p
5478  Lamc1_chr1_12183    2B1  ...         NaN       NaN
5479  Lamc1_chr1_12183    2B2  ...         NaN       NaN
5480  Lamc1_chr1_12183     A1  ...         NaN       NaN
5481  Lamc1_chr1_12183     A2  ...         NaN       NaN
5482  Lamc1_chr1_12183     B1  ...         NaN       NaN
...                ...    ...  ...         ...       ...
5839  Map1b_chr13_9447     B1  ...    0.997996  0.359264
5840  Map1b_chr13_9447     B1  ...    1.108135  0.344776
5841  Map1b_chr13_9447     B1  ...    0.654022  0.421640
5842  Map1b_chr13_9447     B1  ...    0.951995  0.365925
5843  Map1b_chr13_9447     B1  ...    0.643780  0.424082

[366 rows x 10 columns])
kwargs:    {}
Exception: "ValueError('p < 0, p > 1 or p contains NaNs')"


ValueError: p < 0, p > 1 or p contains NaNs

In [ ]:
batch.fit_to_simulations(client)

In [ ]:
batch._flatten_all_parameters()

In [ ]:
batch.plot_nb_spread()

In [ ]:
cluster.close()